# MPOB Palm Oil — Data Exploration

The forecast target is the **Malaysian Palm Oil Board's official daily crude palm oil
price**, "Local Delivered" — the weighted average of actual reported transactions.

This is a *physical* price. It has no contracts, no expiry, and therefore none of the
monthly roll discontinuities that make futures series awkward to forecast. It is
published the next working day, so there is effectively no publication lag.

| | |
|---|---|
| Source | [bepi.mpob.gov.my](https://bepi.mpob.gov.my) |
| Units | MYR per tonne |
| Daily | 4,502 observations, 2008-01-02 → present |
| Weekly | 971 Friday closes, complete grid |

Why not FRED or Yahoo: [`00_FRED_source_evaluation.ipynb`](00_FRED_source_evaluation.ipynb)
and [`DATA.md`](DATA.md). Cutoff selection: [`02_cutoff_selection.ipynb`](02_cutoff_selection.ipynb).

**Prerequisite:** `uv run python scripts/fetch_mpob.py`

Charts are interactive — click a legend entry to hide a series, drag to zoom,
double-click to reset.


---
## 1. Load the series


In [1]:
from __future__ import annotations

import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env")

from cpo.data import MPOB_DAILY_SERIES_ID, MPOB_WEEKLY_SERIES_ID, build_mpob_service
from cpo.plots import (
    DEFAULT_CUTOFFS,
    HORIZONS_WEEKS,
    plot_cutoff_windows,
    plot_news_coverage,
    plot_period_changes,
    plot_price_history,
)


svc = build_mpob_service(cache_dir=ROOT / "data" / "mpob")
as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)

daily = svc.get_series(MPOB_DAILY_SERIES_ID, as_of=as_of)
weekly = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of)

print(f"daily  : {len(daily):,} rows  {daily.timestamp.min():%Y-%m-%d} -> {daily.timestamp.max():%Y-%m-%d}")
print(f"weekly : {len(weekly):,} rows  {weekly.timestamp.min():%Y-%m-%d} -> {weekly.timestamp.max():%Y-%m-%d}")
print(f"price  : RM {daily.value.min():,.0f} to RM {daily.value.max():,.0f} per tonne")
weekly.tail()


daily  : 4,502 rows  2008-01-02 -> 2026-08-07
weekly : 971 rows  2008-01-04 -> 2026-08-07
price  : RM 1,403 to RM 8,076 per tonne


,timestamp,value,released_at
966,2026-07-10,4481.0,2026-07-10
967,2026-07-17,4495.5,2026-07-17
968,2026-07-24,4603.5,2026-07-24
969,2026-07-31,4498.0,2026-07-31
970,2026-08-07,4510.5,2026-08-07


`released_at` equals `timestamp` here, and that is correct — MPOB posts each day's
weighted average the next working day, so the observation is public within a day of
the date it describes. Contrast FRED, where a June price was not published until
13 July and the harness needed an explicit release calendar to avoid leaking it.


---
## 2. Data quality

Three things must hold before this series can be forecast: the weekly grid must be
complete (the harness resolves ground truth by exact timestamp match), there must be
no long gaps, and there must be no contract-roll artifacts.


In [2]:
weekly_series = weekly.set_index("timestamp")["value"]
expected_fridays = pd.date_range(weekly_series.index.min(), weekly_series.index.max(), freq="W-FRI")
gaps = daily.timestamp.diff().dt.days.dropna()

print(
    f"weekly grid   : {len(weekly_series)} of {len(expected_fridays)} expected Fridays, "
    f"{len(set(expected_fridays) - set(weekly_series.index))} missing"
)
print(f"all Fridays   : {set(weekly_series.index.dayofweek) == {4}}")
print(f"largest gap   : {int(gaps.max())} days   (gaps > 10 days: {(gaps > 10).sum()})")
print(f"trading days/yr: {daily.groupby(daily.timestamp.dt.year).size().to_dict()}")

weekly grid   : 971 of 971 expected Fridays, 0 missing
all Fridays   : True
largest gap   : 7 days   (gaps > 10 days: 0)
trading days/yr: {2008: 244, 2009: 248, 2010: 247, 2011: 238, 2012: 241, 2013: 243, 2014: 237, 2015: 242, 2016: 245, 2017: 242, 2018: 244, 2019: 245, 2020: 245, 2021: 244, 2022: 237, 2023: 235, 2024: 240, 2025: 242, 2026: 143}


In [3]:
# Contract-roll check: a futures series jumps on the first trading day of each month.
# A physical price should not.
d = daily.set_index("timestamp")["value"]
returns = (d.pct_change() * 100).dropna()
first_of_month = set(d.groupby(d.index.to_period("M")).head(1).index)
is_first = returns.index.isin(first_of_month)

ratio = returns[is_first].abs().mean() / returns[~is_first].abs().mean()
top20 = returns.abs().nlargest(20).index
print(f"first-of-month |move| : {returns[is_first].abs().mean():.2f}%")
print(f"other days     |move| : {returns[~is_first].abs().mean():.2f}%")
print(f"ratio                 : {ratio:.1f}x   (Yahoo CPO=F futures: 5.2x)")
print(f"top-20 moves on the 1st: {sum(1 for i in top20 if i in first_of_month)}/20   (CPO=F: 19/20)")

first-of-month |move| : 1.61%
other days     |move| : 1.01%
ratio                 : 1.6x   (Yahoo CPO=F futures: 5.2x)
top-20 moves on the 1st: 4/20   (CPO=F: 19/20)


A ratio near 1 means no roll artifact. Yahoo's `CPO=F` scores **5.2x with 19 of its 20
largest daily moves on a roll date** — roughly 3.6% of purely synthetic movement injected
at every month boundary, contaminating one week in four. That is the single reason this
project uses MPOB rather than the futures series.


---
## 3. The signal

Full weekly history since 2008, with the seven selected cutoffs marked.


In [4]:
plot_price_history(
    weekly,
    cutoffs=DEFAULT_CUTOFFS,
    start=None,
    title="MPOB crude palm oil, weekly (Friday close)",
    units="MYR per tonne",
    currency="RM",
    show_blackouts=False,
)

Three regimes are visible:

- **2008–2020** — range-bound, roughly RM 1,800–3,800
- **2021–2022** — the surge to **RM 7,600**, driven by labour shortages and then
  Indonesia's April 2022 export ban, followed by the collapse back below RM 3,500
- **2023–present** — a gradual climb to around RM 4,500

The 2022 spike is the episode FRED never published during. MPOB captured it daily.


---
## 4. Weekly returns

What the model actually has to predict. Blue is up, red is down.


In [5]:
plot_period_changes(weekly, start="2024-01-01", title="MPOB weekly change, %")

In [6]:
weekly_returns = weekly.set_index("timestamp")["value"].pct_change() * 100
recent = weekly_returns["2024-01-01":]
print(f"weekly volatility (2024+) : {recent.std():.2f}%")
print(f"largest weekly move       : {recent.abs().max():.2f}%")
print(f"weeks moving more than 5% : {(recent.abs() > 5).sum()} of {len(recent)}")
recent.reindex(recent.abs().sort_values(ascending=False).index).head(8).round(2).to_frame("pct_change")

weekly volatility (2024+) : 2.81%
largest weekly move       : 7.72%
weeks moving more than 5% : 10 of 136


,pct_change
timestamp,
2024-04-19,-7.72
2026-03-13,7.28
2024-10-25,7.12
2025-04-18,-7.07
2024-12-06,6.68
2024-12-20,-6.27
2026-03-06,5.83
2025-04-11,-5.59


---
## 5. News coverage

An agent can only beat a statistical baseline where there is news to read, so GDELT
volume is a selection criterion for cutoffs — not just background.


In [7]:
articles = pd.read_csv(ROOT / "implementations" / "pko" / "palm_articles_daily.csv", parse_dates=["date"])
print(
    f"{articles.article_count.sum():,} articles over {len(articles):,} days, "
    f"{articles.date.min():%Y-%m-%d} -> {articles.date.max():%Y-%m-%d}"
)

plot_news_coverage(articles, cutoffs=DEFAULT_CUTOFFS)

2,584 articles over 716 days, 2024-01-01 -> 2026-08-10


Coverage is **not uniform**. It collapses from roughly June 2025 to February 2026 —
December 2025 has only 6 articles in the entire month — then recovers strongly through
2026. Cutoffs in the sparse stretch would give an agent nothing to reason over, so
notebook 02 requires a minimum article count in the eight weeks before each cutoff.


---
## 6. The forecast setup

Seven cutoffs, five horizons. Selection logic is in
[`02_cutoff_selection.ipynb`](02_cutoff_selection.ipynb).


In [8]:
print(f"horizons (weeks): {HORIZONS_WEEKS}")

rows = []
for cut in DEFAULT_CUTOFFS:
    visible = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=cut.timestamp.to_pydatetime())
    rows.append(
        {
            "cutoff": cut.date,
            "kind": cut.kind,
            "price_RM": round(float(visible.value.iloc[-1])),
            "weeks_of_history": len(visible),
            "newest_visible": f"{visible.timestamp.max():%Y-%m-%d}",
            "why": cut.label,
        }
    )
pd.DataFrame(rows)

horizons (weeks): [1, 2, 4, 8, 13]


,cutoff,kind,price_RM,weeks_of_history,newest_visible,why
0,2024-04-05,event,4512,849,2024-04-05,-7.7% two weeks out; -8.9% over 13 weeks
1,2024-10-11,event,4402,876,2024-10-11,+7.1% two weeks out; +7.3% over 13 weeks
2,2025-04-04,event,4764,901,2025-04-04,-7.1% two weeks out; -15.4% over 13 weeks
3,2026-02-27,event,3956,948,2026-02-27,+7.3% two weeks out; +13.3% over 13 weeks
4,2025-01-03,quiet,4726,888,2025-01-03,max weekly move ahead 3.7%; flat over 13 weeks
5,2025-06-27,quiet,3956,913,2025-06-27,max weekly move ahead 3.8%
6,2026-05-08,quiet,4515,958,2026-05-08,"calmest window: max 2.4%, flat over 13 weeks"


`newest_visible` equals the cutoff date at every origin — there is no information gap.
Under FRED the newest price was typically two months stale, which made a nominal
one-step forecast a three-month extrapolation.


---
## 7. Cutoff windows

Each shaded band is one cutoff's 13-week forecast window — orange for event, blue for
quiet. This is the visual check that the labels are honest.


In [9]:
plot_cutoff_windows(weekly, cutoffs=DEFAULT_CUTOFFS, horizons=13)

---
## 8. Where this leaves us

| Decision | Status |
|---|---|
| Target | MPOB daily CPO, Local Delivered, MYR/tonne |
| Frequency | Weekly, Friday close — 971 points, complete grid |
| Horizons | 1, 2, 4, 8, 13 weeks |
| Cutoffs | 7 selected — 4 event, 3 quiet |
| Leak safety | No publication lag; `released_at` == `timestamp` |
| Roll artifacts | None — physical price |

**Next:** a naive last-value baseline over these cutoffs, to establish the floor every
other model has to beat.
